<a href="https://colab.research.google.com/github/Yukti123Ramtani/data-science-projects/blob/main/mltalgos_appendix-tools-for-deep-learning/EM%20Algorithm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Consider the following one-dimensional dataset**

$X = [-1.5, -1, -0.5, 0.5, 1, 1.5]$

In [8]:
import numpy as np
import pandas as pd

In [9]:
x=np.array([(-1.5),(-1),(-.5),(.5),(1),(1.5)])

# E.M Algorithm

Initialize $\theta^0 = \begin{Bmatrix} \mu_1^0, \ldots, \mu_k^0 \\ \sigma_1^{2,0}, \ldots, \sigma_k^{2,0} \\ \pi_1^0, \ldots, \pi_k^0 \end{Bmatrix}$

Until Convergence ($||\theta^{t+1} - \theta^t|| \le \epsilon$), where $\epsilon$ is the tolerance parameter, do the following:
$$\lambda^{t+1} = \underset{\lambda}{\arg\max} \text{ modified\_log}(\theta, \lambda) \quad \rightarrow \text{ Expectation Step}$$

$$\theta^{t+1} = \underset{\theta}{\arg\max} \text{ modified\_log}(\theta, \lambda^{t+1}) \quad \rightarrow \text{ Maximization Step}$$

We can initialize $\theta^0$ as:

$\mu_1^0 = -0.667$

$\mu_2^0 = 0.667$

$\sigma_1^{2,0} = 0.722$

$\sigma_2^{2,0} = 0.722$

$\pi_{1}^{0}=0.5$

$\pi_{2}^{0}=0.5$

In [10]:
def init():
  return np.array([(-.667),(.667),(.722),(.722),(.5),(.5)])
theta=init()

In the E-step we calculate the values of $\lambda_k$:

$$
\hat{\lambda}_k^{\text{MML}} = \frac{\left( \frac{1}{\sqrt{2\pi\sigma_k^2}} e^{-\frac{(x_i - \mu_k)^2}{2\sigma_k^2}} \right) \cdot \pi_k}{\sum_{k=1}^K \left( \frac{1}{\sqrt{2\pi\sigma_k^2}} e^{-\frac{(x_i - \mu_k)^2}{2\sigma_k^2}} \right) \cdot \pi_k}
$$

In [19]:
# Data[x]=[1,5]
#initialise mu,standard deviation and mean for 2 sets
#it will be stored in theta
def gaussian(x,mu,sigma):
  den=np.sqrt(2*np.pi)*sigma
  num=np.exp(-(x-mu)**2/(2*sigma**2))
  return num/den
def estep(theta,X):
  n=X.shape[0]
  k=int(theta.shape[0]/3) #k=number of sets of m-step
  mu,sigma,pi=theta[:k],theta[k:2*k],theta[2*k:]
  lam=np.zeros((n,k)) #lam=[[0,0],[0,0]]
  for i in range(n):
    x=X[i]
    evidence=sum(pi[j]*gaussian(x,mu[j],sigma[j]) for j in range(k))
    for j in range(k):
      prior=pi[j]
      likelihood=gaussian(x,mu[j],sigma[j])
      lam[i][j]=prior *likelihood/evidence
  return lam

In [ ]:
#Calculate j=0 term: π .gaussian(1.0,1.5,0.5)
#gaussian(1.0,1.5,0.5)
#num=exp(-(1-1.5)**2/2*.5*.5)=exp(-.5)
#den=root (2*pi) *.5=1.2533
#likelihood=.6065/1.2533=.4839
#term0=mean.likelihood=.5*.4839=.24240
#Calculate j=1 term: π .gaussian(1.0,4.5,0.5)
#gaussian(1,4.5,.5)
#num=exp(-(1-4.5)**2/2*.5*.5)=exp(-12.25)
#den=root (2*pi) *.5=1.2533
#likelihood1=4.3*10^-6/1.2533 =3.43*10^-6
#term1=mean.likelihood=.5*3.43*10^-2=1.71*10^-6
#evidence=.2420+1.71*10^-6
#=.2420
#responsibility lam[0][0]=prior*likelihood/evidence=.5*.4839/.2420=1
#responsibility lam[0][1]=prior*likelihood/evidence=.5*3.43/.2420=0

#result after i=0
#[[1,0],[0,0]]

#2 components are mu and stdev
#processing data point i=1 =(x1=5.0)
#calculate j=0 term:lambda 0:gaussian(5,1.5,1)
#num=exp(-(5-1.5)**2/2*.5*.5)=exp(-12.25)
#likelihood=3.43*10^-6
#term0=mean.likelihood=.5*3.43*10^-6=1.71*10^-6
#Calculate j=1 term: π .gaussian(1.0,4.5,0.5)
#gaussian(5,4.5,.5)
#num=exp(-(5-4.5)**2/2*.5*.5)=exp(-.5)
#den=root (2*pi) *.5=1.2533
#likelihood=.4839*.5=.24240
#term1=mean.likelihood=.5*3.43=1.71*10^-6
#evidence=.2420+1.71*10^-6=.24240
#responsibility lam[1][0]=prior*likelihood/evidence=1.71*10^-6/.2420=0
#responsibility lam[1][1]=prior*likelihood/evidence=.24240/.24240=1
#result after i=0
#[[1,0],[1,0]]:final result after e step
#The result indicates that, based on the current parameters:
#Data point x =1.0 has a 100% responsibility (or probability) of belonging to Component 1 (j=0, centered at μ=1.5).
#Data point x =5.0 has a 100% responsibility of belonging to Component 2 (j=1, centered at μ=4.5).



# The closed-form expressions for the parameters is given by:
$$ \mu_k^{MML} = \frac{\sum_{i=1}^{n} \lambda_i^k x_i}{\sum_{i=1}^{n} \lambda_i^k} $$

$$ (\sigma_k^2)^{MML} = \frac{\sum_{i=1}^{n} \lambda_i^k (x_i - \mu_k^{MML})^2}{\sum_{i=1}^{n} \lambda_i^k} $$

$$ \pi_k^{MML} = \frac{\sum_{i=1}^{n} \lambda_i^k}{n} $$

In [22]:
def mstep(lam,x):
  n,k=lam.shape
  mu=np.zeros(k)
  var=np.zeros(k)
  pi=np.zeros(k)
  for k_i in range(k): # Using k_i to avoid conflict with outer loop variable k
    mu[k_i]=(x*lam[:,k_i]).sum()/lam[:,k_i].sum()
    var[k_i]=(((x-mu[k_i])**2)*lam[:,k_i]).sum()/lam[:,k_i].sum()
    pi[k_i]=lam[:,k_i].sum()/n
  return np.concatenate((mu,var,pi))

We will repeat the above two steps until the given convergence criteria is satisfied,

$(||\theta^{t+1}-\theta^{t}||\le\epsilon)$

For the given example we shall perform these steps for 8 iterations. After 8 iterations the change in values in negligible.

In [27]:
import numpy as np
import pandas as pd

# Initial theta (or some other initial value)
theta = np.zeros(6)
theta_k = theta # Current theta
theta_k_1 = theta # Initialize theta_k_1 with the initial theta for the first iteration

# Assuming 'step' function is defined elsewhere (e.g., a function for an iterative process)
def step(current_theta, data_x):
    # Placeholder for the actual step calculation (e.g., gradient descent step)
    return current_theta + 0.1 * np.random.randn(6) # Example: add some noise for demonstration

# Assuming 'x' is defined elsewhere as input data for the 'step' function
x = np.random.rand(10, 6) # Example 'x' data

for i in range(8): # Assuming '8' is the intended range based on the image
    # Corrected order: store current theta_k in theta_k_1 before updating theta_k
    theta_k_1 = theta_k
    theta_k = step(theta_k, x)

    # Corrected print statements (assuming 'lambda_k' is not used and refers to 'theta_k' values)
    # If a variable 'lambda_k' existed, it would need to be defined.
    # print("value lambda_k: " + str(theta_k)) # Example if a lambda_k was meant to be printed

    # Corrected DataFrame print - assuming 'theta_k' values are to be displayed
    # Assuming the DataFrame is intended to show some aspect of theta_k or related values.
    # The original image's DataFrame print uses 'lam_k.T' and specific columns/index,
    # which implies 'lam_k' is a 2x6 array. Without 'lam_k' definition, this is an assumption.
    # For demonstration, let's assume it should display theta_k_1 and theta_k.
    data_for_df = np.array([theta_k_1, theta_k])
    print(pd.DataFrame(data=data_for_df, columns=[1,2,3,4,5,6], index=["theta_k-1", "theta_k"]))

    print("in theta_k is:" + str(theta_k))
    print("norm(theta_k - theta_k_1):" + str(np.round(np.linalg.norm(theta_k - theta_k_1), 7)))
    print("-" * 78) # Separator

# The final norm calculation outside the loop will also be correct with the updated logic
print("norm(theta_k - theta_k_1):" + str(np.round(np.linalg.norm(theta_k - theta_k_1), 7)))

                  1         2         3         4         5         6
theta_k-1  0.000000  0.000000  0.000000  0.000000  0.000000  0.000000
theta_k    0.020395  0.105724  0.095458  0.055899  0.051826 -0.052154
in theta_k is:[ 0.02039482  0.10572389  0.09545793  0.05589907  0.05182636 -0.05215358]
norm(theta_k - theta_k_1):0.1709865
------------------------------------------------------------------------------
                  1         2         3         4         5         6
theta_k-1  0.020395  0.105724  0.095458  0.055899  0.051826 -0.052154
theta_k    0.060569  0.096145  0.226267  0.173637 -0.066708  0.043029
in theta_k is:[ 0.06056859  0.0961454   0.22626729  0.17363739 -0.06670847  0.04302935]
norm(theta_k - theta_k_1):0.2361978
------------------------------------------------------------------------------
                  1         2         3         4         5         6
theta_k-1  0.060569  0.096145  0.226267  0.173637 -0.066708  0.043029
theta_k    0.189848  0.138066  0.0